### create new cache on hard drive

In [1]:
from pathlib import Path
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

drive = Path("/Volumes/WD_BLACK")

cache_dir = drive / "Spikes2Slope" / "allen_visual_coding_data"
cache_dir.mkdir(parents=True, exist_ok=True)

manifest_path = cache_dir / "manifest.json"

coding_cache = EcephysProjectCache.from_warehouse(
    manifest=str(manifest_path)
)

print("Cache directory:", cache_dir)
print("Manifest exists:", manifest_path.exists())
print("Available storage:",
      round(__import__("shutil").disk_usage(drive).free / 1024**3, 2),
      "GiB")


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cache directory: /Volumes/WD_BLACK/Spikes2Slope/allen_visual_coding_data
Manifest exists: True
Available storage: 369.55 GiB


### identify optotagging-eligible sessions

In [2]:
import numpy as np
import pandas as pd

coding_sessions = coding_cache.get_session_table()

opto_sessions = (
    coding_sessions
    .rename_axis("ecephys_session_id")
    .reset_index()
    [["ecephys_session_id", "specimen_id", "full_genotype"]]
    .copy()
)

genotype = opto_sessions["full_genotype"].fillna("")

opto_sessions["cre_line"] = np.select(
    [
        genotype.str.startswith("Pvalb"),
        genotype.str.startswith("Sst"),
        genotype.str.startswith("Vip")
    ],
    ["Pvalb", "Sst", "Vip"],
    default="other"
)

eligible = opto_sessions.loc[
    (opto_sessions["cre_line"] != "other")
    & genotype.str.contains("Ai32", regex=False)
].copy()

eligible = eligible.sort_values(
    ["cre_line", "ecephys_session_id"]
).reset_index(drop=True)

assert len(eligible) == 28, "Expected 28 eligible sessions"
assert eligible["ecephys_session_id"].is_unique

print("Eligible sessions:", len(eligible))
print("\nSessions by Cre line:")
print(eligible["cre_line"].value_counts().sort_index())

display(eligible)

eligible.to_csv(
    cache_dir.parent / "eligible_opto_sessions.csv",
    index=False
)


Eligible sessions: 28

Sessions by Cre line:
Pvalb     8
Sst      12
Vip       8
Name: cre_line, dtype: int64


,ecephys_session_id,specimen_id,full_genotype,cre_line
0,721123822,707296982,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
1,746083955,726170935,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
2,760345702,739783171,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
3,773418906,757329624,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
4,797828357,776061251,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
5,829720705,811322619,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
6,839557629,821469666,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
7,840012044,820866121,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Pvalb
8,715093703,699733581,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Sst
9,719161530,703279284,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,Sst


### the big download

In [3]:

import gc
import time

download_log_path = cache_dir.parent / "opto_download_log.csv"

if download_log_path.exists():
    download_log = pd.read_csv(download_log_path)
else:
    download_log = pd.DataFrame(columns=[
        "ecephys_session_id", "cre_line",
        "status", "error", "elapsed_seconds"
    ])

for row in eligible.itertuples(index=False):
    session_id = int(row.ecephys_session_id)

    completed = download_log.loc[
        (download_log["ecephys_session_id"] == session_id)
        & (download_log["status"] == "success")
    ]

    if not completed.empty:
        print(f"Skipping {session_id}: already downloaded")
        continue

    print(f"\nDownloading session {session_id} ({row.cre_line})...")
    start = time.time()

    try:
        session = coding_cache.get_session_data(session_id)

        epochs = session.optogenetic_stimulation_epochs
        n_units = len(session.units)

        status = "success"
        error = ""

        print(
            f"Success: {session_id} | "
            f"{n_units} units | {len(epochs)} opto epochs"
        )

        del session
        gc.collect()

    except Exception as exc:
        status = "failed"
        error = str(exc)
        print(f"Failed: {session_id} | {error}")

    elapsed = round(time.time() - start, 1)

    download_log = download_log[
        download_log["ecephys_session_id"] != session_id
    ]

    download_log = pd.concat([
        download_log,
        pd.DataFrame([{
            "ecephys_session_id": session_id,
            "cre_line": row.cre_line,
            "status": status,
            "error": error,
            "elapsed_seconds": elapsed
        }])
    ], ignore_index=True)

    download_log.to_csv(download_log_path, index=False)

print("\nDownload summary:")
display(download_log["status"].value_counts())

print("\nLog saved to:", download_log_path)


Downloading: 100%|██████████| 1.74G/1.74G [01:46<00:00, 16.3MB/s]
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs

Success: 721123822 | 444 units | 180 opto epochs



Downloading: 100%|██████████| 2.56G/2.56G [02:51<00:00, 14.9MB/s]
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs

Success: 746083955 | 582 units | 180 opto epochs



Downloading: 100%|██████████| 1.96G/1.96G [03:02<00:00, 10.7MB/s]
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs

...

Success: 819701982 | 585 units | 300 opto epochs



Downloading: 100%|██████████| 2.01G/2.01G [02:30<00:00, 13.3MB/s]
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs

Success: 835479236 | 582 units | 300 opto epochs

Download summary:


success    28
Name: status, dtype: int64


Log saved to: /Volumes/WD_BLACK/Spikes2Slope/opto_download_log.csv


### successfully downloaded now to inspect optogenetic stimulation protocols

In [4]:

import gc

protocol_results = []

for row in eligible.itertuples(index=False):
    session_id = int(row.ecephys_session_id)
    session = coding_cache.get_session_data(session_id)
    epochs = session.optogenetic_stimulation_epochs

    pulses = epochs.loc[
        (epochs["stimulus_name"] == "pulse")
        & epochs["duration"].between(
            0.009, 0.020, inclusive="neither"
        )
    ]

    protocol_results.append({
        "ecephys_session_id": session_id,
        "cre_line": row.cre_line,
        "total_epochs": len(epochs),
        "ten_ms_pulses": len(pulses),
        "light_levels": sorted(pulses["level"].dropna().unique())
    })

    del session
    gc.collect()

protocol_summary = pd.DataFrame(protocol_results)

display(protocol_summary)

print("Sessions analyzed:", len(protocol_summary))
print("Sessions with 10 ms pulses:",
      (protocol_summary["ten_ms_pulses"] > 0).sum())

print("Sessions without 10 ms pulses:",
      (protocol_summary["ten_ms_pulses"] == 0).sum())

protocol_summary.to_csv(
    cache_dir.parent / "opto_protocol_summary.csv",
    index=False
)


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

,ecephys_session_id,cre_line,total_epochs,ten_ms_pulses,light_levels
0,721123822,Pvalb,180,45,"[1.0, 2.5, 4.0]"
1,746083955,Pvalb,180,45,"[1.0, 2.5, 4.0]"
2,760345702,Pvalb,180,45,"[1.0, 2.5, 4.0]"
3,773418906,Pvalb,180,45,"[1.0, 2.5, 4.0]"
4,797828357,Pvalb,300,75,"[1.3, 1.7, 2.0]"
5,829720705,Pvalb,300,75,"[1.3, 1.7, 2.0]"
6,839557629,Pvalb,300,75,"[1.3, 1.7, 2.0]"
7,840012044,Pvalb,300,75,"[0.6375, 0.7375, 0.82]"
8,715093703,Sst,180,45,"[1.0, 2.5, 4.0]"
9,719161530,Sst,180,45,"[1.0, 2.5, 4.0]"


Sessions analyzed: 28
Sessions with 10 ms pulses: 28
Sessions without 10 ms pulses: 0


### optotagging responses across sessions

In [5]:

visual_regions = [
    "VISp", "VISam", "VISrl", "VISal",
    "VISl", "VISpm", "LGd"
]

results_dir = cache_dir.parent / "opto_results"
results_dir.mkdir(exist_ok=True)

all_results = []
processing_log = []

for row in eligible.itertuples(index=False):
    session_id = int(row.ecephys_session_id)
    result_path = results_dir / f"opto_units_{session_id}.csv"

    if result_path.exists():
        all_results.append(pd.read_csv(result_path))
        print(f"Loaded saved results: {session_id}")
        continue

    try:
        session = coding_cache.get_session_data(session_id)
        epochs = session.optogenetic_stimulation_epochs

        pulses = epochs.loc[
            (epochs["stimulus_name"] == "pulse")
            & epochs["duration"].between(
                0.009, 0.020, inclusive="neither"
            )
        ]

        if pulses.empty:
            raise ValueError("No compatible 10 ms pulses")

        trial_starts = pulses["start_time"].to_numpy()

        units = session.units.loc[
            session.units["ecephys_structure_acronym"].isin(
                visual_regions
            )
        ]

        session_results = []

        for unit_id, unit in units.iterrows():
            spikes = np.asarray(session.spike_times[unit_id])

            baseline_counts = (
                np.searchsorted(spikes, trial_starts - 0.002)
                - np.searchsorted(spikes, trial_starts - 0.010)
            )

            evoked_counts = (
                np.searchsorted(spikes, trial_starts + 0.009)
                - np.searchsorted(spikes, trial_starts + 0.001)
            )

            baseline_rate = baseline_counts.mean() / 0.008
            evoked_rate = evoked_counts.mean() / 0.008

            session_results.append({
                "unit_id": unit_id,
                "ecephys_session_id": session_id,
                "specimen_id": row.specimen_id,
                "cre_line": row.cre_line,
                "region": unit["ecephys_structure_acronym"],
                "baseline_rate_hz": baseline_rate,
                "evoked_rate_hz": evoked_rate,
                "response_ratio": evoked_rate / (baseline_rate + 1),
                "snr": unit["snr"],
                "amplitude_cutoff": unit["amplitude_cutoff"],
                "presence_ratio": unit["presence_ratio"],
                "isi_violations": unit["isi_violations"],
                "n_pulses": len(pulses)
            })

        session_df = pd.DataFrame(session_results)

        session_df["candidate_tagged"] = (
            session_df["response_ratio"] > 2
        )

        session_df["allen_qc"] = (
            (session_df["presence_ratio"] >= 0.95)
            & (session_df["isi_violations"] <= 0.5)
            & (session_df["amplitude_cutoff"] <= 0.1)
        )

        session_df["snr_qc"] = (
            session_df["allen_qc"]
            & (session_df["snr"] >= 1)
        )

        session_df["matched_candidate"] = (
            session_df["candidate_tagged"]
            & session_df["snr_qc"]
        )

        assert session_df["unit_id"].is_unique

        session_df.to_csv(result_path, index=False)
        all_results.append(session_df)

        processing_log.append({
            "ecephys_session_id": session_id,
            "status": "success",
            "units": len(session_df),
            "candidate_tagged": int(
                session_df["candidate_tagged"].sum()
            ),
            "qc_matched": int(
                session_df["matched_candidate"].sum()
            )
        })

        print(
            f"{session_id} ({row.cre_line}): "
            f"{len(session_df)} units, "
            f"{session_df['matched_candidate'].sum()} QC candidates"
        )

    except Exception as exc:
        processing_log.append({
            "ecephys_session_id": session_id,
            "status": "failed",
            "error": str(exc)
        })
        print(f"FAILED {session_id}: {exc}")

    finally:
        if "session" in locals():
            del session
        gc.collect()

opto_results = pd.concat(all_results, ignore_index=True)
processing_log = pd.DataFrame(processing_log)

assert opto_results["unit_id"].is_unique

opto_results.to_csv(
    results_dir / "all_opto_unit_results.csv",
    index=False
)

processing_log.to_csv(
    results_dir / "processing_log.csv",
    index=False
)

print("\nSessions with results:",
      opto_results["ecephys_session_id"].nunique())

print("Total candidate optotagged units:",
      int(opto_results["candidate_tagged"].sum()))

print("QC-matched candidates:",
      int(opto_results["matched_candidate"].sum()))


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

721123822 (Pvalb): 190 units, 22 QC candidates


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

746083955 (Pvalb): 247 units, 32 QC candidates


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

...

831882777 (Sst): 411 units, 46 QC candidates


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

839068429 (Sst): 287 units, 16 QC candidates


/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.6.0-alpha is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/Users/keanuventura/miniforge3/envs/allen_env/lib/python3.10/site-packages/hdmf/spec/namespace.py:531

KeyboardInterrupt: 